In [ ]:
import nanonispy2 as nap2
import nanonispy as nap
import nanonispyfit 
import numpy as np 
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import pandas as pd 
import pickle 
import cv2 

from skimage.color import rgb2gray, label2rgb
from skimage.feature import blob_log, peak_local_max
from skimage import exposure, filters
from skimage.util import img_as_float
from skimage.morphology import erosion, dilation, opening, closing, disk, square, white_tophat
from skimage.filters import threshold_otsu
from skimage.segmentation import clear_border, watershed
from skimage.measure import label, regionprops, find_contours
from skimage.restoration import (
    denoise_tv_chambolle,
    denoise_bilateral,
    denoise_wavelet,
    estimate_sigma,
)
from skimage import img_as_float
from skimage.util import random_noise
from skimage.metrics import peak_signal_noise_ratio


from scipy import misc # denoising the imaage 
from scipy import ndimage as ndi

from PIL import Image



def denoise_images(im):
    """
    Uses cv2 to denoise image
    
    fast NlMeansDenoising 
    
    Making the image clearer, less blurry. 
    Args:
        im: The image to be processed, grayscaled
    Return: 
        denoised_im: processed image 
    """
    im_float = img_as_float(im)
    im_rescaled = exposure.rescale_intensity(im_float, in_range='image', out_range=(0, 1))
    im_8u = (im_rescaled * 255).astype(np.uint8)
    denoised_im = cv2.fastNlMeansDenoising(im_8u, h=10)
    return denoised_im 

def mathematical_morphology(im):
    """
    Uses mathematical morphology to make the border of the image clearer
    
    Args: 
        im: The image to be processed
    Return: 
        gradient: processed image
    """
    selem = disk(1)

    eroded = erosion(im, selem)
    dilated = dilation(im, selem)

    # Opening and closing
    opened = opening(im, selem)
    closed = closing(im, selem)

    gradient = dilation(im, selem) - erosion(im, selem)
    return gradient

def draw_contours(im):
    # Step 1: Flatten background with white tophat
    im_flat = white_tophat(im, footprint=disk(30))

    # Step 2: Apply Otsu threshold on the flattened image
    thresh = threshold_otsu(im_flat)
    bw = closing(im_flat > thresh, square(3))

    # Step 3: Remove artifacts connected to the border
    cleared = clear_border(bw)

    # Step 4: Label image regions
    label_image = label(cleared)
    # Step 5: Overlay segmentation on the original image
    image_label_overlay = label2rgb(label_image, image=im, bg_label=0, alpha=0.4)

    fig, ax = plt.subplots(figsize=(10, 6))
    ax.imshow(image_label_overlay, cmap='gray')

    # Draw contours instead of rectangles
    for region in regionprops(label_image):
        if region.area >= 50:  # keep only sufficiently large objects
            # get binary mask for this region
            region_mask = (label_image == region.label)
            # extract contour(s)
            contours = find_contours(region_mask, level=0.5)
            for contour in contours:
                ax.plot(contour[:, 1], contour[:, 0], color='red', linewidth=2)
    ax.set_axis_off()
    plt.title(f"Number of Molecules: {sum(1 for region in regionprops(label_image) if region.area >= 50)}")
    plt.tight_layout()
    plt.show()
    

def enhance_image_for_segmentation(image):
    """
    Comprehensive preprocessing pipeline for SXM images
    """
    # Step 1: Normalize the image
    image_norm = (image - image.min()) / (image.max() - image.min())

    image_denoised = filters.gaussian(image_norm, sigma=1)
    
    image_contrast = exposure.equalize_adapthist(image_denoised, clip_limit=0.03)
    
    background_subtracted = white_tophat(image_contrast, disk(15))

    edges = filters.laplace(background_subtracted)
    enhanced = background_subtracted + 0.5 * edges
    enhanced = (enhanced - enhanced.min()) / (enhanced.max() - enhanced.min())
    
    return enhanced

def smart_watershed_segmentation(image, border_remove=10):
    """
    Enhanced watershed segmentation with optimal preprocessing
    """
    # Remove borders first
    if border_remove > 0:
        image = image[border_remove:-border_remove, border_remove:-border_remove]
    
    # Apply comprehensive preprocessing
    enhanced_image = enhance_image_for_segmentation(image)
    
    # Create binary mask using adaptive thresholding
    threshold = filters.threshold_otsu(enhanced_image)
    binary_mask = enhanced_image > threshold
    
    # Clean up the binary mask
    binary_mask = ndi.binary_closing(binary_mask, structure=disk(1))
    binary_mask = ndi.binary_opening(binary_mask, structure=disk(1))
    
    # Distance transform
    distance = ndi.distance_transform_edt(binary_mask)
    
    # Find peaks with optimized parameters
    coords = peak_local_max(
        distance,
        min_distance=5,  # Minimum distance between peaks
        threshold_abs=0.2 * distance.max(),  # Relative threshold
        footprint=np.ones((3, 3)),
        exclude_border=2  # Avoid border peaks
    )
    
    # Create markers
    marker_mask = np.zeros(distance.shape, dtype=bool)
    if len(coords) > 0:
        marker_mask[tuple(coords.T)] = True
    
    markers, _ = ndi.label(marker_mask)
    
    # Apply watershed
    labels = watershed(-distance, markers, mask=binary_mask)
    
    return enhanced_image, binary_mask, labels

def visualize_processing_pipeline(original, enhanced, binary, labels):
    """
    Visualize each step of the processing pipeline
    """
    fig, axes = plt.subplots(2, 3, figsize=(15, 10))
    
    # Original image
    axes[0,0].imshow(original, cmap='gray')
    axes[0,0].set_title('Original Image', fontsize=12, weight='bold')
    axes[0,0].axis('off')
    
    # Enhanced image
    axes[0,1].imshow(enhanced, cmap='gray')
    axes[0,1].set_title('Enhanced Image', fontsize=12, weight='bold')
    axes[0,1].axis('off')
    
    # Binary mask
    axes[0,2].imshow(binary, cmap='gray')
    axes[0,2].set_title('Binary Mask', fontsize=12, weight='bold')
    axes[0,2].axis('off')
    
    # Distance transform
    axes[1,0].imshow(-distance, cmap='viridis')
    axes[1,0].set_title('Distance Transform', fontsize=12, weight='bold')
    axes[1,0].axis('off')
    
    # Markers
    axes[1,1].imshow(markers > 0, cmap='hot')
    axes[1,1].set_title('Detected Markers', fontsize=12, weight='bold')
    axes[1,1].axis('off')
    
    # Final segmentation
    axes[1,2].imshow(labels, cmap='nipy_spectral')
    num_molecules = len(np.unique(labels)) - 1
    axes[1,2].set_title(f'Segmentation: {num_molecules} molecules', 
                       fontsize=12, weight='bold')
    axes[1,2].axis('off')
    
    plt.tight_layout()
    plt.show()
    
    return num_molecules

